# NB15-T6b — Deletion rule and the test-baseline projection (SPEC decision 37)

On test our flags are already ~63% real (row 0: P 0.634, R 0.703); on Extra-dev ~21%. A verifier tuned on dev
removes too many real errors at the test's base rate. This notebook (CPU, a few minutes, reads caches only):

1. scores the **deletion rule** (revert every deletion, keep every substitution and insertion; no parameters) on
   BASE, the best single checkpoint, the ensemble and stage-1, with paired sentence-bootstrap intervals;
2. recomputes **row 4** from the saved verifier at the frozen τ 0.10 (must reproduce F1 0.4708);
3. projects every candidate to the test baseline from its dev keep fractions and applies the **headline rule
   fixed in SPEC decision 37 before this run**. No τ or setting is tuned here.

**Run settings:** Accelerator **None**, Internet *On*. Add data: `iqra-cache`, the NB11 output, the NB14-T6 output,
the NB14 stage-1 output and the **NB15-T6 output**. No secrets needed.
**After the run:** Save Version, paste back the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Inputs --------------------------------------------------------------------------------------
import json, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from iqra import data, ensemble as E, posteriors, scorer, transfer as X, verifier
from iqra.config import Config, find_cache_root, find_input
from iqra.vocab import Vocab

cfg = Config()
cfg.paths.cache_root = find_cache_root()
SPLITS_DIR = find_input("splits.json", what="the NB11 output")
T6_DIR = find_input("nb14_t6_report.json", what="the NB14-T6 output")
S1_DIR = find_input("post_stage1_extra_dev.pt", what="the NB14 output (stage-1 caches)")
V_DIR = find_input("nb15_t6_report.json", what="the NB15-T6 output")
OUT = "/kaggle/working/nb15_t6b"
os.makedirs(OUT, exist_ok=True)
vocab = Vocab.load(cfg.paths.vocab)
ID2TOK, BLANK = vocab.id2tok, vocab.blank_id
truth_dev = pd.read_csv(f"{SPLITS_DIR}/truth_extra_dev.csv", keep_default_na=False)
CLUSTERS = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"]))
EXP = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"].str.split()))
t6 = json.load(open(f"{V_DIR}/nb15_t6_report.json"))
report = {"repo_commit": REPO_COMMIT, "test_baseline": X.TEST_BASELINE}
print("inputs:", T6_DIR, S1_DIR, V_DIR, sep="\n  ")


In [ ]:
# ---- Plain decodes on Extra-dev ---------------------------------------------------------------------
caches, info = {}, {}
for path in sorted(__import__("glob").glob(f"{T6_DIR}/post_*_extra_dev.pt")):
    meta, clips = posteriors.read_cache(path)
    caches[meta["checkpoint"]] = clips
    info[meta["checkpoint"]] = (meta["run"], float(meta["t5_f1"]))
six = [n for n, (r, _) in info.items() if r in ("A", "B")]
base = [n for n, (r, _) in info.items() if r == "BASE"][0]
best = max(six, key=lambda n: info[n][1])
ens = E.ensemble_split({n: caches[n] for n in six}, six, ID2TOK, BLANK)
s1 = posteriors.read_cache(f"{S1_DIR}/post_stage1_extra_dev.pt")[1]
PLAIN = {"BASE": {c: e["tokens"] for c, e in caches[base].items()},
         "best_single": {c: e["tokens"] for c, e in caches[best].items()},
         "ensemble6": {c: e["tokens"] for c, e in ens.items()},
         "stage1": {c: e["tokens"] for c, e in s1.items()}}

def counts_of(tok):
    pred = pd.DataFrame({"ID": list(tok), "Prediction": [" ".join(t) for t in tok.values()]})
    return pred, scorer.per_clip_counts(truth_dev, pred)

rows, COUNTS = [], {}
for name, tok in PLAIN.items():
    p0, c0 = counts_of(tok)
    p1, c1 = counts_of({c: X.deletion_rule(EXP[c], t) for c, t in tok.items()})
    off = scorer.official_score(truth_dev, p1, keep_dir=f"{OUT}/score_{name}_deletion_rule")
    assert int(c1["TR"].sum()) == off["TR"] and int(c1["FR"].sum()) == off["FR"]
    p1.to_csv(f"{OUT}/pred_{name}_deletion_rule_extra_dev.csv", index=False)
    g = scorer.paired_bootstrap_gain(c0, c1, CLUSTERS)
    k = X.keep_fractions(c0, c1)
    COUNTS[name] = (c0, c1)
    rows.append({"system": name, "f1_plain": round(scorer.summed_metrics(c0)["f1"], 4), "f1_deletion_rule": off["f1"],
                 "gain": round(g["gain"], 4), "ci90": [round(g["lo"], 4), round(g["hi"], 4)],
                 "excludes_zero": g["excludes_zero"], "keep_tr": round(k["keep_tr"], 3), "keep_fr": round(k["keep_fr"], 3)})
del_table = pd.DataFrame(rows)
print(del_table.to_string(index=False))
report["deletion_rule"] = del_table.to_dict(orient="records")


In [ ]:
# ---- Row 4 recomputed from the saved verifier (frozen tau 0.10) ------------------------------------
train = pd.concat([pd.read_parquet(f"{V_DIR}/candidates_ens_{s}.parquet") for s in ("fold_a", "fold_b", "iqra_dev_slice")],
                  ignore_index=True)
dev_c = pd.read_parquet(f"{V_DIR}/candidates_ens_extra_dev.parquet")
feats = verifier.FEATURE_SETS["primary"]
enc = verifier.Encoder(feats).fit(train[train["op"].isin(verifier.PRIMARY_OPS)].reset_index(drop=True))
booster = lgb.Booster(model_file=f"{V_DIR}/verifier_primary_ensemble.txt")
probs = verifier.predict_primary(booster, enc, dev_c)
TAU = t6["ladder"]["row4_ens_primary_verifier"]["tau"]
c3 = COUNTS["ensemble6"][0]
proj_rows = []
for tau in sorted(set(cfg.verifier.tau_grid) | {TAU}):
    new = verifier.apply(dev_c, probs, EXP, PLAIN["ensemble6"], tau, revert_deletions=True)
    _, c4 = counts_of(new)
    if tau == TAU:
        f1_4 = scorer.summed_metrics(c4)["f1"]
        assert round(f1_4, 4) == t6["ladder"]["row4_ens_primary_verifier"]["f1"], (f1_4, "row 4 not reproduced")
    k = X.keep_fractions(c3, c4)
    pr = X.projected(**k)
    proj_rows.append({"tau": tau, "dev_f1": round(scorer.summed_metrics(c4)["f1"], 4), **{kk: round(v, 3) for kk, v in k.items()},
                      "proj_test_p": round(pr["precision"], 3), "proj_test_r": round(pr["recall"], 3), "proj_test_f1": round(pr["f1"], 4)})
tau_proj = pd.DataFrame(proj_rows)
print(f"row 4 reproduced at tau {TAU}: F1 {f1_4:.4f}")
print("row 4 by tau, projected to the test baseline (information only; tau stays frozen):")
print(tau_proj.to_string(index=False))
tau_proj.to_csv(f"{OUT}/row4_tau_projection.csv", index=False)
report["row4_tau_projection"] = tau_proj.to_dict(orient="records")


In [ ]:
# ---- Decision 37: projected test F1 of each candidate and the headline -------------------------------
dr = {r["system"]: r for r in rows}
r4 = tau_proj[tau_proj["tau"] == TAU].iloc[0]
cands = [
    {"candidate": "row 0 plain", "qualifies": True, "keep_tr": 1.0, "keep_fr": 1.0, "evidence": "baseline"},
    {"candidate": "row 0 + deletion rule", "qualifies": bool(dr["BASE"]["excludes_zero"] and dr["BASE"]["gain"] > 0),
     "keep_tr": dr["BASE"]["keep_tr"], "keep_fr": dr["BASE"]["keep_fr"], "evidence": f"BASE + rule on dev: {dr['BASE']['gain']:+.4f} {dr['BASE']['ci90']}"},
    {"candidate": "row 3 + deletion rule", "qualifies": bool(dr["ensemble6"]["excludes_zero"] and dr["ensemble6"]["gain"] > 0),
     "keep_tr": dr["ensemble6"]["keep_tr"], "keep_fr": dr["ensemble6"]["keep_fr"], "evidence": f"ensemble + rule on dev: {dr['ensemble6']['gain']:+.4f} {dr['ensemble6']['ci90']}"},
    {"candidate": f"row 4 (tau {TAU})", "qualifies": True, "keep_tr": float(r4["keep_tr"]), "keep_fr": float(r4["keep_fr"]),
     "evidence": "T6 pre-registered final system"},
]
for c in cands:
    pr = X.projected(c["keep_tr"], c["keep_fr"])
    c.update({k: round(v, 4) for k, v in pr.items()})
table = pd.DataFrame(cands).rename(columns={"precision": "proj_test_p", "recall": "proj_test_r", "f1": "proj_test_f1"})
with pd.option_context("display.width", 200, "display.max_colwidth", 60):
    print(table.to_string(index=False))
headline = table[table["qualifies"]].sort_values("proj_test_f1", ascending=False, kind="stable").iloc[0]["candidate"]
print(f"\n>>> decision 37 headline for T8: {headline}")
print("(projection = dev keep fractions applied to row 0's test P 0.634 / R 0.703; a heuristic. Every row is still scored once on test.)")
report.update({"candidates": table.to_dict(orient="records"), "headline": headline})
with open(f"{OUT}/nb15_t6b_report.json", "w") as f:
    json.dump(report, f, indent=1, default=str)


In [ ]:
# ---- PASTE BACK -----------------------------------------------------------------------------------------
print(json.dumps(report, indent=1, default=str))
